Code link for 2D optimization: http://blog.sws9f.org/computer-vision/2017/09/07/colorization-using-optimization-python.html

Code link for Gatys Style Transfer: https://pytorch.org/tutorials/advanced/neural_style_tutorial.html

In [6]:
import os
import sys
import tqdm
import copy
import torch
import colorsys
import logging

import numpy as np
import torch.nn as nn
import skimage.io as io
import scipy.misc as smi
import SimpleITK as sitk
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image
from pyamg import solve
from skimage import exposure
from torch.autograd import Variable
from skimage.transform import resize
from skimage import color, img_as_float
from scipy.sparse.linalg import spsolve
from scipy.sparse import csc_matrix, csr_matrix

ModuleNotFoundError: No module named 'pyamg'

In [ ]:
%pip install matplotlib


In [ ]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
imsize = (256,256) if torch.cuda.is_available() else (32,32)

In [ ]:
OUTPUT_DIR = './New_Results_WACV/temp/'
if(not os.path.exists(OUTPUT_DIR)):
    os.makedirs(OUTPUT_DIR)

np.set_printoptions(precision=8, suppress=True)

# Inputs

In [ ]:
path_volume = './generated_test16.mhd'
path_style_image = './style_images/style_image_4.png'

# Gatys Neural Style Transfer Code

#### Image Loader

In [ ]:
loader = transforms.Compose([
    transforms.Resize(imsize),  # scale imported image
    transforms.ToTensor()])  # transform it into a torch tensor


def image_loader(image_name):
    image = Image.open(image_name)
    img = np.array(image)
    # fake batch dimension required to fit network's input dimensions
    image = loader(image).unsqueeze(0)
    return image.to(device, torch.float), img.shape

unloader = transforms.ToPILImage()  # reconvert into PIL image

def imsave(tensor, title=None):
    image = tensor.cpu().clone()  # we clone the tensor to not do changes on it
    image = image.squeeze(0)      # remove the fake batch dimension
    image = unloader(image)
    return np.array(image)

#### Loss Functions

In [ ]:
class ContentLoss(nn.Module):
    def __init__(self, target,):
        super(ContentLoss, self).__init__()
        # we 'detach' the target content from the tree used
        # to dynamically compute the gradient: this is a stated value,
        # not a variable. Otherwise the forward method of the criterion
        # will throw an error.
        self.target = target.detach()

    def forward(self, input):
        self.loss = F.mse_loss(input, self.target)
        return input
    
def gram_matrix(input):
    a, b, c, d = input.size()  # a=batch size(=1)
    # b=number of feature maps
    # (c,d)=dimensions of a f. map (N=c*d)

    features = input.view(a * b, c * d)  # resise F_XL into \hat F_XL

    G = torch.mm(features, features.t())  # compute the gram product

    # we 'normalize' the values of the gram matrix
    # by dividing by the number of element in each feature maps.
    return G.div(a * b * c * d)

class StyleLoss(nn.Module):
    def __init__(self, target_feature):
        super(StyleLoss, self).__init__()
        self.target = gram_matrix(target_feature).detach()

    def forward(self, input):
        G = gram_matrix(input)
        self.loss = F.mse_loss(G, self.target)
        return input
    
class RGB2YUV(nn.Module):
    def __init__(self):
        super(RGB2YUV, self).__init__()
    
    def forward(self, input_img):
        yuv = Variable(torch.zeros_like(input_img)) 
        yuv[:,0,:,:] = input_img[:,0,:,:]*0.2999+ (0.587 * input_img[:,1,:,:])  + 0.114 * input_img[:,2,:,:] 
        yuv[:,1,:,:] = input_img[:,0,:,:]*(-0.14713)+ (-0.28886 * input_img[:,1,:,:])  + 0.436 * input_img[:,2,:,:] 
        yuv[:,2,:,:] = input_img[:,0,:,:]*0.615 + (-0.51499 * input_img[:,1,:,:])  + (-0.10001 * input_img[:,2,:,:] )
        
        return yuv    
    
class TVLoss(nn.Module):
    def __init__(self):
        super(TVLoss, self).__init__()

    def forward(self, input_img):
        return torch.sum(torch.abs(input_img[:,:,:,:-1] - input_img[:,:,:,1:])) + torch.sum(torch.abs(input_img[:,:,:-1,:] - input_img[:,:,1:,:]))

#### Initialize convolution layers

In [ ]:
class Normalization(nn.Module):
    def __init__(self, mean, std):
        super(Normalization, self).__init__()
        # .view the mean and std to make them [C x 1 x 1] so that they can
        # directly work with image Tensor of shape [B x C x H x W].
        # B is batch size. C is number of channels. H is height and W is width.
        self.mean = torch.tensor(mean).view(-1, 1, 1)
        self.std = torch.tensor(std).view(-1, 1, 1)

    def forward(self, img):
        # normalize img
        return (img - self.mean) / self.std
    
content_layers_default = ['conv_{}'.format(i) for i in range(1,8)]
style_layers_default = ['conv_{}'.format(i) for i in range(1,8)]
def get_style_model_and_losses(cnn, normalization_mean, normalization_std,
                               style_img, content_img,
                               content_layers=content_layers_default,
                               style_layers=style_layers_default):
    cnn = copy.deepcopy(cnn)

    # normalization module
    normalization = Normalization(normalization_mean, normalization_std).to(device)

    # just in order to have an iterable access to or list of content/syle
    # losses
    content_losses = []
    style_losses = []

    # assuming that cnn is a nn.Sequential, so we make a new nn.Sequential
    # to put in modules that are supposed to be activated sequentially
    model = nn.Sequential(normalization)

    i = 0  # increment every time we see a conv
    for layer in cnn.children():
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = 'conv_{}'.format(i)
        elif isinstance(layer, nn.ReLU):
            name = 'relu_{}'.format(i)
            # The in-place version doesn't play very nicely with the ContentLoss
            # and StyleLoss we insert below. So we replace with out-of-place
            # ones here.
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d):
            name = 'pool_{}'.format(i)
        elif isinstance(layer, nn.BatchNorm2d):
            name = 'bn_{}'.format(i)
        else:
            raise RuntimeError('Unrecognized layer: {}'.format(layer.__class__.__name__))

        model.add_module(name, layer)

        if name in content_layers:
            # add content loss:
            target = model(content_img).detach()
            content_loss = ContentLoss(target)
            model.add_module("content_loss_{}".format(i), content_loss)
            content_losses.append(content_loss)

        if name in style_layers:
            # add style loss:
            target_feature = model(style_img).detach()
            style_loss = StyleLoss(target_feature)
            model.add_module("style_loss_{}".format(i), style_loss)
            style_losses.append(style_loss)

    # now we trim off the layers after the last content and style losses
    for i in range(len(model) - 1, -1, -1):
        if isinstance(model[i], ContentLoss) or isinstance(model[i], StyleLoss):
            break

    model = model[:(i + 1)]

    return model, style_losses, content_losses

#### Optimizer

In [ ]:
def get_input_optimizer(input_img):
    # this line to show that input is a parameter that requires a gradient
    optimizer = optim.LBFGS([input_img.requires_grad_()], lr=1.0e0)
    return optimizer

In [ ]:
def run_style_transfer(cnn, normalization_mean, normalization_std,
                       content_img, style_img, input_img, num=0, num_steps=300,
                       style_weight=1000000, content_weight=1, tv_weight=2e-2, diff_weight=1e-1):
    """Run the style transfer."""
    print('Building the style transfer model..')
    model, style_losses, content_losses = get_style_model_and_losses(cnn,
        normalization_mean, normalization_std, style_img, content_img)
    optimizer = get_input_optimizer(input_img)
    rgb2yuv = RGB2YUV().to(device)
    criterion_tv = TVLoss().to(device)

    print('Optimizing..')
    run = [0]
    while run[0] <= num_steps:

        def closure():
            # correct the values of updated input image
            input_img.data.clamp_(0, 1)

            optimizer.zero_grad()
            model(input_img)
            
            rgb2yuv_img = rgb2yuv(input_img)
            tv_loss = tv_weight * (torch.sum(torch.abs(rgb2yuv_img[:, 1:3 , :, :-1] - rgb2yuv_img[:, 1:3 ,:, 1:]))\
                                   + torch.sum(torch.abs(rgb2yuv_img[:, 1:3 , :-1, :] - rgb2yuv_img[:, 1:3 , 1:,:])))
            tv_loss_content_img = tv_weight * criterion_tv(content_img)
            diff_loss = torch.abs(tv_loss - tv_loss_content_img)
            total_loss = tv_loss + diff_weight * diff_loss
            
            total_loss.backward(retain_graph=True)
            
            style_score = 0
            content_score = 0

            for sl in style_losses:
                style_score += sl.loss
            for cl in content_losses:
                content_score += cl.loss

            style_score *= style_weight
            content_score *= content_weight

            loss = style_score + content_score
            loss.backward()

            run[0] += 1
            if run[0] % 50 == 0:
                print("run {}:".format(run))
                print('Style Loss : {:4f} Content Loss: {:4f}, TV Loss:{:.4f} Diff Loss: {:.4f} '.format(
                    style_score.item(), content_score.item(), tv_loss.item(), diff_loss.item()))
                print()
                output_img = resize(imsave(input_img), input_img.shape[2:])
#                 io.imsave(OUTPUT_DIR + 'test_{}_{}.png'.format(num, run), output_img)

            return style_score + content_score

        optimizer.step(closure)

    # a last correction...
    input_img.data.clamp_(0, 1)

    return input_img

# Preprocessing

In [ ]:
def read_numpy_volume(path):
    vol = sitk.ReadImage(path)
    np_vol = sitk.GetArrayFromImage(vol)

    # np_vol = (np_vol - np.min(np_vol)) / (np.max(np_vol) - np.min(np_vol))

    return np_vol

def to_grey(np_vol):
    if(len(np_vol.shape)==4):
        h,ww,l,c = np_vol.shape
    else:
        h,ww,l = np_vol.shape
    
    grey = np.zeros((h,ww,l,3))
    
    for i in range(h):
        if(len(np_vol.shape)==4):
            grey_img = color.rgb2gray(np_vol[i,:,:])
        else:
            grey_img = np_vol[i,:,:]
        
        if(np.max(grey_img)>1):
            grey_img = grey_img / 255.0
        
        grey[i,:,:,0], grey[i,:,:,1], grey[i,:,:,2] = grey_img, grey_img, grey_img
    
    return grey

def generate_hints(grey_vol, color_vol, offset=10, axis='x', save=True):
    h,ww,l,c = grey_vol.shape
    out_vol = np.zeros((h,ww,l,c))
    out_vol = grey_vol.copy()
    
    cnn = models.vgg19(pretrained=True).features.to(device).eval()

    cnn_normalization_mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
    cnn_normalization_std = torch.tensor([0.229, 0.224, 0.225]).to(device)

    slices = []
    if(axis=='x'):
        num_hints = h // offset
        for i in range(num_hints):
            slices.append(grey_vol[i*offset,:,:,:])
    elif(axis=='y'):
        num_hints = ww // offset
        for i in range(num_hints):
            slices.append(grey_vol[:,i*offset,:,:])
    else:
        num_hints = l // offset
        for i in range(num_hints):
            slices.append(grey_vol[:,:,i*offset,:])

    for i in tqdm.trange(num_hints):
        curr_slices = slices[i]
        
        if save:
            io.imsave(OUTPUT_DIR+'/l_channel.png', curr_slices)
            io.imsave(OUTPUT_DIR+'/l_channel_{}.png'.format(i), curr_slices)

        content_img, content_shape = image_loader(OUTPUT_DIR+'l_channel.png')
        style_img, _ = image_loader(path_style_image)
        
        input_img = content_img.clone()

        output = run_style_transfer(cnn, cnn_normalization_mean, cnn_normalization_std,
                                    content_img, style_img, input_img, num=i, num_steps=500, 
                                content_weight=2.5, tv_weight=2e-1, diff_weight=0)

        output_img = resize(imsave(output), content_shape[:2])
        io.imsave(OUTPUT_DIR+'test_{}.png'.format(i), output_img)

        if(axis=='x'):
            out_vol[i*offset,:,:,:] = img_as_float(io.imread(OUTPUT_DIR+'test_{}.png'.format(i)))
        elif(axis=='y'):
            out_vol[:,i*offset,:,:] = img_as_float(io.imread(OUTPUT_DIR+'test_{}.png'.format(i)))
        else:
            out_vol[:,:,i*offset,:] = img_as_float(io.imread(OUTPUT_DIR+'test_{}.png'.format(i)))
    
    return out_vol

# Optimization Code

In [ ]:
class WindowNeighbor:
    def __init__(self, width, center, pic):
        # center is a list of [row, col, Y_intensity]
        self.center = [center[0], center[1], center[2], pic[center][0]]
        self.width = width
        self.neighbors = None
        self.find_neighbors(pic)
        self.mean = None
        self.var = None

    def find_neighbors(self, pic):
        self.neighbors = []
        ix_r_min = max(0, self.center[0] - self.width)
        ix_r_max = min(pic.shape[0], self.center[0] + self.width + 1)
        ix_c_min = max(0, self.center[1] - self.width)
        ix_c_max = min(pic.shape[1], self.center[1] + self.width + 1)
        ix_l_min = max(0, self.center[2] - self.width)
        ix_l_max = min(pic.shape[2], self.center[2] + self.width + 1)
        for r in range(ix_r_min, ix_r_max):
            for c in range(ix_c_min, ix_c_max):
                for l in range(ix_l_min, ix_l_max):
                    if r == self.center[0] and c == self.center[1] and l ==self.center[2]:
                        continue
                    self.neighbors.append([r,c,l,pic[r,c,l,0]])

    def __str__(self):
        return 'windows c=(%d, %d, %f) size: %d' % (self.center[0], self.center[1], self.center[2], len(self.neighbors))

# affinity functions, calculate weights of pixels in a window by their intensity.
def affinity_a(w):
    nbs = np.array(w.neighbors)
    sY = nbs[:,3]
    cY = w.center[3]
    diff = sY - cY
    sig = np.var(np.append(sY, cY))
    if sig < 1e-6:
        sig = 1e-6  
    wrs = np.exp(- np.power(diff,2) / (sig * 2.0))
    wrs = - wrs / np.sum(wrs)
    nbs[:,3] = wrs
    return nbs

# translate (row,col) to/from sequential number
def to_seq(i,j,k, grey_vol):
    index = np.ravel_multi_index([int(i),int(j),int(k)], grey_vol[:,:,:,0].shape)
    return index

def fr_seq(seq, rows):
    r = seq % rows
    c = int((seq - r) / rows)
    return (r, c)

# combine 3 channels of YUV to a RGB photo: n x n x n x 3 array
def yuv_channels_to_rgb(cY,cU,cV, h, ww, l):
    ansRGB = [colorsys.yiq_to_rgb(cY[i],cU[i],cV[i]) for i in range(len(cY))]
    ansRGB = np.array(ansRGB)
    pic_ansRGB = np.zeros((h, ww, l, 3))
    pic_ansRGB[:,:,:,0] = ansRGB[:,0].reshape(h, ww, l, order='C')
    pic_ansRGB[:,:,:,1] = ansRGB[:,1].reshape(h, ww, l, order='C')
    pic_ansRGB[:,:,:,2] = ansRGB[:,2].reshape(h, ww, l, order='C')
    return pic_ansRGB

### Run preprocessing

In [ ]:
off = 5
ax = 'x'

In [ ]:
color_vol = read_numpy_volume(path_volume)

grey_vol = to_grey(read_numpy_volume(path_volume))

print('The shape of input volume = {}'.format(grey_vol.shape))

g_vol = sitk.GetImageFromArray(grey_vol)
g_vol = sitk.WriteImage(g_vol, OUTPUT_DIR+'g.mhd')

hints_vol = generate_hints(grey_vol, color_vol, offset=off, axis=ax)
h,ww,l,c = grey_vol.shape

In [ ]:
plt.imshow(hints_vol[45,:,:]); plt.show()

In [ ]:
plt.imshow(grey_vol[45,:,:]); plt.show()

### Initialize optimization parameters

#### Parameter for class WindowWidth

In [ ]:
wd_width = 1

#### Create weight matrix A and vectors b_u and b_v

In [ ]:
pic_size = h*ww*l

channel_Y,_,_ = colorsys.rgb_to_yiq(grey_vol[:,:,:,0],grey_vol[:,:,:,1],grey_vol[:,:,:,2])
_,channel_U,channel_V = colorsys.rgb_to_yiq(hints_vol[:,:,:,0],hints_vol[:,:,:,1],hints_vol[:,:,:,2])

map_colored = (abs(channel_U) + abs(channel_V)) > 0.0001

pic_yuv = np.zeros((h,ww,l,3))
pic_yuv[:,:,:,0] = channel_Y
pic_yuv[:,:,:,1] = channel_U
pic_yuv[:,:,:,2] = channel_V
weightData = []
num_pixel_bw = 0

for i in tqdm.trange(h):
    for j in range(ww):
        for k in range(l):
            res = []
            w = WindowNeighbor(wd_width, (i,j,k), pic_yuv)
            if( not map_colored[i,j,k]):
                weights = affinity_a(w)
                for e in weights:
                    weightData.append([w.center, (e[0],e[1],e[2]), e[3]])
            weightData.append([w.center, (w.center[0],w.center[1], w.center[2]), 1.])

In [ ]:
sp_idx_rc_data = [[to_seq(e[0][0],e[0][1],e[0][2],grey_vol), to_seq(e[1][0],e[1][1],e[1][2],grey_vol), e[2]] for e in weightData]
sp_idx_rc = np.array(sp_idx_rc_data, dtype=np.integer)[:,0:2]
sp_data = np.array(sp_idx_rc_data, dtype=np.float64)[:,2]

matA = csr_matrix((sp_data, (sp_idx_rc[:,0], sp_idx_rc[:,1])), shape=(pic_size, pic_size))

b_u = np.zeros(pic_size)
b_v = np.zeros(pic_size)
idx_colored = np.nonzero(map_colored.reshape(pic_size, order='C'))

pic_u_flat = pic_yuv[:,:,:,1].reshape(pic_size, order='C')
b_u[idx_colored] = pic_u_flat[idx_colored]

pic_v_flat = pic_yuv[:,:,:,2].reshape(pic_size, order='C')
b_v[idx_colored] = pic_v_flat[idx_colored]

### Run optimization

In [ ]:
ansY = pic_yuv[:,:,:,0].reshape(pic_size, order='C')
ansU = solve(matA, b_u, tol=1.0e-10, verb=True)
ansV = solve(matA, b_v, tol=1.0e-10, verb=True)

In [ ]:
pic_ans = yuv_channels_to_rgb(ansY,ansU,ansV, h, ww, l)

In [ ]:
sitk.WriteImage(sitk.GetImageFromArray(pic_ans), 't_out.mhd')

# Post processing

In [ ]:
pic_ans = sitk.GetArrayFromImage(sitk.ReadImage('./t_out.mhd'))

In [ ]:
def post_process(grey_vol, color_vol, offset=10, axis='x'):
    h,ww,l,c = grey_vol.shape
    out_vol = np.zeros((h,ww,l,c))
    out_vol = color_vol.copy()
    
    slices = []
    if(axis=='x'):
        num_hints = h // offset
        for i in range(num_hints):
            slices.append(grey_vol[i*offset,:,:,:])
    elif(axis=='y'):
        num_hints = ww // offset
        for i in range(num_hints):
            slices.append(grey_vol[:,i*offset,:,:])
    else:
        num_hints = l // offset
        for i in range(num_hints):
            slices.append(grey_vol[:,:,i*offset,:])

    for i in tqdm.trange(num_hints):
        if(axis=='x'):
            out_vol[i*offset,:,:,:] = slices[i]
        elif(axis=='y'):
            out_vol[:,i*offset,:,:] = slices[i]
        else:
            out_vol[:,:,i*offset,:] = slices[i]
            
    pic_size = h*ww*l

    channel_Y,_,_ = colorsys.rgb_to_yiq(grey_vol[:,:,:,0],grey_vol[:,:,:,1],grey_vol[:,:,:,2])
    _,channel_U,channel_V = colorsys.rgb_to_yiq(out_vol[:,:,:,0],out_vol[:,:,:,1],out_vol[:,:,:,2])

    map_colored = (abs(channel_U) + abs(channel_V)) > 0.0001

    pic_yuv = np.zeros((h,ww,l,3))
    pic_yuv[:,:,:,0] = channel_Y
    pic_yuv[:,:,:,1] = channel_U
    pic_yuv[:,:,:,2] = channel_V
    weightData = []
    num_pixel_bw = 0

    for i in tqdm.trange(h):
        for j in range(ww):
            for k in range(l):
                res = []
                w = WindowNeighbor(wd_width, (i,j,k), pic_yuv)
                if( not map_colored[i,j,k]):
                    weights = affinity_a(w)
                    for e in weights:
                        weightData.append([w.center, (e[0],e[1],e[2]), e[3]])
                weightData.append([w.center, (w.center[0],w.center[1], w.center[2]), 1.])
                
    sp_idx_rc_data = [[to_seq(e[0][0],e[0][1],e[0][2],grey_vol), to_seq(e[1][0],e[1][1],e[1][2],grey_vol), e[2]] for e in weightData]
    sp_idx_rc = np.array(sp_idx_rc_data, dtype=np.integer)[:,0:2]
    sp_data = np.array(sp_idx_rc_data, dtype=np.float64)[:,2]

    matA = csr_matrix((sp_data, (sp_idx_rc[:,0], sp_idx_rc[:,1])), shape=(pic_size, pic_size))

    b_u = np.zeros(pic_size)
    b_v = np.zeros(pic_size)
    idx_colored = np.nonzero(map_colored.reshape(pic_size, order='C'))

    pic_u_flat = pic_yuv[:,:,:,1].reshape(pic_size, order='C')
    b_u[idx_colored] = pic_u_flat[idx_colored]

    pic_v_flat = pic_yuv[:,:,:,2].reshape(pic_size, order='C')
    b_v[idx_colored] = pic_v_flat[idx_colored]
    
    ansY = pic_yuv[:,:,:,0].reshape(pic_size, order='C')
    ansU = solve(matA, b_u, tol=1.0e-10, verb=False)
    ansV = solve(matA, b_v, tol=1.0e-10, verb=False)

    out_vol = yuv_channels_to_rgb(ansY,ansU,ansV, h, ww, l)
    
    return out_vol

In [ ]:
post_processed_vol = post_process(grey_vol, pic_ans, off, ax)

### Save the generated volume

In [ ]:
sitk.WriteImage(sitk.GetImageFromArray(post_processed_vol), OUTPUT_DIR+'post_process_1.mhd')

# Final Generated Volume

In [ ]:
sitk.WriteImage(sitk.GetImageFromArray(exposure.adjust_gamma(post_processed_vol, 1.5)), 
                OUTPUT_DIR+'final_output.mhd')